##### `========== GENERATING THE SYNTHETIC ENERGY DATASET ==============` #####

###### **`IMPORT THE NECESSARY LIBRARIES`**

In [1]:
import numpy as np
import pandas as pd
import os

###### **`DEFINE THE PARAMETERS FOR THE ENERGY DATASET`**

In [2]:
np.random.seed(10)

no_of_customers = 100

# HOURLY READING FOR 60 DAYS
no_of_days = 60
readings_per_day = 24
n_rows = no_of_customers * no_of_days * readings_per_day

# BASE LEVEL CUSTOMER INFORMATION
customer_ids = np.arange(201, no_of_customers + 201)
regions = ['North', 'South', 'East', 'West']
tariff_plans = ['fixed', 'variable']

# CREATE CUSTOMER ENERGY DATAFRAME
customer_df = pd.DataFrame({
    'customer_id': customer_ids,
    'region': np.random.choice(regions, size=no_of_customers, p=[0.25, 0.35, 0.2, 0.2]),
    'tariff_plan': np.random.choice(tariff_plans, size=no_of_customers, p=[0.6, 0.4]),
    'is_smart_meter': np.random.choice([0, 1], size=no_of_customers, p=[0.3, 0.7])
})

customer_df.head()

,customer_id,region,tariff_plan,is_smart_meter
0,201,East,fixed,1
1,202,North,variable,1
2,203,East,fixed,1
3,204,East,fixed,1
4,205,South,variable,1


In [3]:
np.random.seed(10)

# EXPAND THE DATETIME RANGE FOR THE CUSTOMER ENERGY DATASET FOR TIME SERIES
date_range = pd.date_range('2026-01-01', periods=no_of_days * readings_per_day, freq='h')
full_index = pd.MultiIndex.from_product(
    [customer_ids, date_range],
    names=['customer_id', 'timestamp_utc']
)

date_df = full_index.to_frame(index=False)
date_df.head()


,customer_id,timestamp_utc
0,201,2026-01-01 00:00:00
1,201,2026-01-01 01:00:00
2,201,2026-01-01 02:00:00
3,201,2026-01-01 03:00:00
4,201,2026-01-01 04:00:00


In [4]:
# MERGE CUSTOMER ATTRIBUTES WITH THE EXPANDED DATE RANGE

cust_df = date_df.merge(customer_df, on='customer_id', how='left')
cust_df.head()

,customer_id,timestamp_utc,region,tariff_plan,is_smart_meter
0,201,2026-01-01 00:00:00,East,fixed,1
1,201,2026-01-01 01:00:00,East,fixed,1
2,201,2026-01-01 02:00:00,East,fixed,1
3,201,2026-01-01 03:00:00,East,fixed,1
4,201,2026-01-01 04:00:00,East,fixed,1


In [5]:
np.random.seed(10)
# SIMULATE KILOWATTS-HOUR (kwh) CONSUMPTION

# SKEWED DISTRIBUTION TO REFLECT REALISTIC ENERGY USAGE PATTERNS - POSITIVELY SKEWED
base_kwh = np.random.gamma(shape=2.0, scale=0.7, size=len(cust_df))

# ADD DAILY AND HOURLY PATERNS TO THE BASE KWH
hour_of_day = cust_df['timestamp_utc'].dt.hour

# ADD PEAK USAGE IN THE EVENING HOURS (6 PM TO 10 PM)
cust_df['kwh'] = base_kwh * (1 + 0.3 * ((hour_of_day >= 18) & (hour_of_day <= 22)))

# ADD REGIONAL DIFFERENCES
region_factor = cust_df['region'].map({
    'North': 1.1,
    'South': 0.9,
    'East': 1.0,
    'West': 1.05
})
cust_df['kwh'] *= region_factor

# OUTAGE MINUTES IN LAST 24 HOURS BASED ON REGION WITH NORTH AND EAST HAVING MORE OUTAGES
cust_df['outage_minutes_last_24h'] = np.random.poisson(
    lam=cust_df['region'].map({'North': 15, 'South': 5, 'East': 10, 'West': 7})
)


# AGGREGATE TO MONTHLY-ISH BILL AMOUNT - APPROXIMATE PER READING -> ASSUME PRICE PER KWH DEPENDS ON TARIFF AND REGION
price_per_kwh = (
    cust_df['tariff_plan'].map({'fixed': 0.18, 'variable': 0.16}) *
    cust_df['region'].map({'North': 1.05, 'South': 0.95, 'East': 1.0, 'West': 1.02})
)
cust_df['bill_amount_eur'] = cust_df['kwh'] * price_per_kwh


# SIMULATE COMPLAINTS - BASED ON OUTAGES, TARIFF PLAN, AND REGION -> HIGHER PROBABILITY OF COMPLAINTS FOR VARIABLE TARIFF, HIGH OUTAGES, AND NORTH REGION
base_prob = 0.02
cust_df['complaint_prob'] = base_prob + 0.03 * (cust_df['tariff_plan'] == 'variable') + 0.02 * (cust_df['region'] == 'North') + 0.0005 * cust_df['outage_minutes_last_24h']

cust_df['complaint_prob'] = cust_df['complaint_prob'].clip(0, 0.5)
cust_df['complaint_flag'] = np.random.binomial(1, cust_df['complaint_prob'])

# METER_ID - UNIQUE IDENTIFIER FOR THE METER, SIMULATED AS 'MID' + CUSTOMER_ID
cust_df['meter_id'] = 'MID' + cust_df['customer_id'].astype(str)


# REORDER COLUMNS FOR BETTER READABILITY
cust_df = cust_df[[
    'customer_id', 'region', 'meter_id', 'timestamp_utc', 'kwh',
    'tariff_plan', 'is_smart_meter', 'outage_minutes_last_24h',
    'bill_amount_eur', 'complaint_flag'
]]


# FINAL DATASET WITH MESSINESS INJECTED
df = cust_df.copy()

# RANDOMLY DROP SOME REGIONS TO SIMULATE MISSING DATA
mask_missing_region = np.random.rand(len(df)) < 0.02
df.loc[mask_missing_region, 'region'] = np.nan

# NEGATIVE KWH VALUES ATTRIBUTED TO SENSOR ERRORS
mask_negative_kwh = np.random.rand(len(df)) < 0.005
df.loc[mask_negative_kwh, 'kwh'] *= -1

# DUPLICATE SOME ROWS TO SIMULATE DUPLICATION
duplicates = df.sample(frac=0.01, random_state=42)
df_messy = pd.concat([df, duplicates], ignore_index=True)

# CREATE OUTLIER BILLS - SOME BILLS ARE 10 TIMES HIGHER THAN EXPECTED
mask_outlier_bill = np.random.rand(len(df_messy)) < 0.003
df_messy.loc[mask_outlier_bill, 'bill_amount_eur'] *= 10

# FINAL MESSY DATASET READY FOR ANALYSIS
energy_data = df_messy.copy()
print(energy_data.shape)
energy_data.head()

(145440, 10)


,customer_id,region,meter_id,timestamp_utc,kwh,tariff_plan,is_smart_meter,outage_minutes_last_24h,bill_amount_eur,complaint_flag
0,201,East,MID201,2026-01-01 00:00:00,2.831160,fixed,1,15,0.509609,0
1,201,East,MID201,2026-01-01 01:00:00,1.939790,fixed,1,10,0.349162,0
2,201,East,MID201,2026-01-01 02:00:00,1.823063,fixed,1,11,0.328151,0
3,201,East,MID201,2026-01-01 03:00:00,0.629419,fixed,1,7,0.113295,0
4,201,NaN,MID201,2026-01-01 04:00:00,1.423434,fixed,1,10,0.256218,0


##### `SAVE THE FINAL MESSY DATASET TO DATA/RAW`

In [13]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_path = os.path.join(BASE_DIR, "data", "raw", "energy_data.csv")

os.makedirs(os.path.dirname(data_path), exist_ok=True)

energy_data.to_csv(data_path, index=False)

#### `LOAD THE SYTHENTIC ENERGY DATASET INTO THE MYSQL DATABASE`

In [6]:
import yaml
from sqlalchemy import create_engine, text

In [7]:

# FUNCTION TO LOAD DATABASE CONFIGURATION FROM YAML FILE AND CONNECT TO MYSQL DATABASE -> YOU WILL MANUALLY CREATE THE DATABASE IN THE MYSQL SERVER FIRST
'''def load_db_config(config_path="../config/mysql_db_config.yaml"):
    with open(config_path, "r") as file:
        config = yaml.safe_load(file)
    return config["mysql"]

def get_mysql_engine(config_path="../config/mysql_db_config.yaml"):
    config = load_db_config(config_path)
    uri = (
        f"mysql+pymysql://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/{config['database']}"
    )
    engine = create_engine(uri)
    return engine
    '''

'def load_db_config(config_path="../config/mysql_db_config.yaml"):\n    with open(config_path, "r") as file:\n        config = yaml.safe_load(file)\n    return config["mysql"]\n\ndef get_mysql_engine(config_path="../config/mysql_db_config.yaml"):\n    config = load_db_config(config_path)\n    uri = (\n        f"mysql+pymysql://{config[\'user\']}:{config[\'password\']}"\n        f"@{config[\'host\']}:{config[\'port\']}/{config[\'database\']}"\n    )\n    engine = create_engine(uri)\n    return engine\n    '

In [ ]:
# FUNCTION TO LOAD DATABASE CONFIGURATION FROM YAML FILE AND CONNECT TO MYSQL DATABASE -> THIS FUNCTION ALSO CHECKS IF THE DATABASE EXISTS AND CREATES IT IF IT DOES NOT EXIST
def load_db_config(config_path="../config/mysql_db_config.yaml"):
    with open(config_path, "r") as file:
        config = yaml.safe_load(file)
    return config["mysql"]

def get_mysql_engine(config_path="../config/mysql_db_config.yaml"):
    config = load_db_config(config_path)
    # CREATE A TEMPORARY ENGINE WITHOUT THE DATABASE TO ENSURE THE DATABASE EXISTS
    uri = (
        f"mysql+pymysql://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}"
    )
    temp_engine = create_engine(uri)

    # CREATE DATABASE IF NOT EXISTS
    with temp_engine.connect() as connection:
        connection.execute(text(f"CREATE DATABASE IF NOT EXISTS {config['database']}"))
    
    # NOW CREATE THE ENGINE WITH THE DATABASE INCLUDED
    full_uri = f"{uri}/{config['database']}"
    engine = create_engine(full_uri)
    
    return engine

In [9]:
# INSTANTIATE THE DATABASE ENGINE AND THE CONFIGURATION
mysql_engine = get_mysql_engine()
db_config = load_db_config()
database_name = db_config["database"]

# print(f"Database Engine: {mysql_engine}")
print(f"Database Name: {database_name}")

Database Name: energy_analytics_db


In [10]:
# LOAD DATAFRAME TO MYSQL DATABASE
table_name = "energy_consumption_data"

# LOAD THE DATAFRAME TO MYSQL DATABASE AFTER YOU HAVE CREATED THE DATABASE IN THE MYSQL SERVER
'''energy_data.to_sql(table_name, mysql_engine, if_exists="replace", index=False)
print(f"Data successfully loaded into the data table name: {table_name} in the database: {database_name}")'''

'energy_data.to_sql(table_name, mysql_engine, if_exists="replace", index=False)\nprint(f"Data successfully loaded into the data table name: {table_name} in the database: {database_name}")'

In [11]:
# LOAD THE DATAFRAME TO MYSQL DATABASE IN CHUNKS TO HANDLE LARGE DATASETS -> THIS IS MORE EFFICIENT AND PREVENTS MEMORY ISSUES
from math import ceil

CHUNK_SIZE = 5000
NO_CHUNKS = ceil(len(energy_data) / CHUNK_SIZE)

for k in range(NO_CHUNKS):
    CHUNK = energy_data.iloc[k*CHUNK_SIZE:(k+1)*CHUNK_SIZE]

    CHUNK.to_sql(table_name, mysql_engine, if_exists="append" if k > 0 else "replace", index=False, method="multi")

    print(f"Chunk {k+1}/{NO_CHUNKS} loaded successfully.")

Chunk 1/30 loaded successfully.
Chunk 2/30 loaded successfully.
Chunk 3/30 loaded successfully.
Chunk 4/30 loaded successfully.
Chunk 5/30 loaded successfully.
Chunk 6/30 loaded successfully.
Chunk 7/30 loaded successfully.
Chunk 8/30 loaded successfully.
Chunk 9/30 loaded successfully.
Chunk 10/30 loaded successfully.
Chunk 11/30 loaded successfully.
Chunk 12/30 loaded successfully.
Chunk 13/30 loaded successfully.
Chunk 14/30 loaded successfully.
Chunk 15/30 loaded successfully.
Chunk 16/30 loaded successfully.
Chunk 17/30 loaded successfully.
Chunk 18/30 loaded successfully.
Chunk 19/30 loaded successfully.
Chunk 20/30 loaded successfully.
Chunk 21/30 loaded successfully.
Chunk 22/30 loaded successfully.
Chunk 23/30 loaded successfully.
Chunk 24/30 loaded successfully.
Chunk 25/30 loaded successfully.
Chunk 26/30 loaded successfully.
Chunk 27/30 loaded successfully.
Chunk 28/30 loaded successfully.
Chunk 29/30 loaded successfully.
Chunk 30/30 loaded successfully.


In [12]:
# READ THE DATA FROM MYSQL DATABASE TO THE NOTEBOOK
pd.read_sql(f"SELECT * FROM {table_name} LIMIT 10;", mysql_engine)

,customer_id,region,meter_id,timestamp_utc,kwh,tariff_plan,is_smart_meter,outage_minutes_last_24h,bill_amount_eur,complaint_flag
0,201,East,MID201,2026-01-01 00:00:00,2.831160,fixed,1,15,0.509609,0
1,201,East,MID201,2026-01-01 01:00:00,1.939790,fixed,1,10,0.349162,0
2,201,East,MID201,2026-01-01 02:00:00,1.823063,fixed,1,11,0.328151,0
3,201,East,MID201,2026-01-01 03:00:00,0.629419,fixed,1,7,0.113295,0
4,201,NaN,MID201,2026-01-01 04:00:00,1.423434,fixed,1,10,0.256218,0
5,201,East,MID201,2026-01-01 05:00:00,1.267537,fixed,1,10,0.228157,0
6,201,East,MID201,2026-01-01 06:00:00,1.603374,fixed,1,10,0.288607,0
7,201,East,MID201,2026-01-01 07:00:00,2.626516,fixed,1,8,0.472773,0
8,201,East,MID201,2026-01-01 08:00:00,1.385716,fixed,1,10,0.249429,0
9,201,East,MID201,2026-01-01 09:00:00,1.616942,fixed,1,13,0.291049,0
